# 17 · Paddy Doctor — descomposición ANOVA descriptiva
**Complemento al bootstrap · sin GPU ni entrenamiento adicional**

Este notebook reproduce el análisis ya ejecutado a partir del ZIP con las 120 corridas del notebook 15. Por cada dominio objetivo, descompone la variación observada de macro-F1 de control, macro-F1 de transferencia y pérdida de macro-F1.

El diseño es balanceado: **4 configuraciones × 5 emparejamientos × 3 semillas**. Se conservan efectos principales y todas sus interacciones. No se calculan pruebas F, valores p ni componentes de varianza poblacional: hay una puntuación por combinación, cero grados de libertad residuales en el modelo completo y solapamiento entre emparejamientos.

Selecciona **CPU** y «Ejecutar todas». El bootstrap 16 se conserva como evidencia pareada de incertidumbre por imagen y sensibilidad por semilla. La descomposición ANOVA es un diagnóstico complementario.


## 1. Conectar Drive y localizar los resultados
No se modifican los resultados del notebook 15 ni los del bootstrap 16.


In [ ]:
from google.colab import drive
from pathlib import Path
import sys, subprocess, json, shutil

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/PADDY_DOCTOR_Q1')
INPUT_ZIP = PROJECT / 'exports/15_P5_Rematching_5x3_30ep_v1_RESULTADOS.zip'
OUTPUT = PROJECT / 'results/17_P5_ANOVA_Descriptiva_v1'
if not INPUT_ZIP.is_file(): raise FileNotFoundError(INPUT_ZIP)
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Insumo:', INPUT_ZIP)
print('Salida:', OUTPUT)


## 2. Dependencias y código incorporado
Las tablas se verifican contra las predicciones originales sobre los mismos IDs de test. La suma de cuadrados de cada término se comprueba mediante una transformación ortonormal de Helmert independiente. También hay pruebas sintéticas de efectos puros e interacciones.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'scikit-learn', 'matplotlib'], check=True)
import importlib.util
MODULE_TEXT = '"""Balanced factorial sum-of-squares decomposition for completed P5 runs.\nDescriptive ANOVA only: no F tests, p-values or random-effect variance estimates.\n"""\nfrom pathlib import Path\nfrom itertools import combinations,product\nfrom zipfile import ZipFile,ZIP_DEFLATED\nimport argparse,json,hashlib,tempfile,platform,math,csv\nimport numpy as np\nimport pandas as pd\n\nMODELS=[\'resnet50\',\'convnext_tiny\',\'vit_small_patch16_224\',\'swin_tiny_patch4_window7_224\']\nLABELS={\'resnet50\':\'ResNet-50\',\'convnext_tiny\':\'ConvNeXt-Tiny\',\'vit_small_patch16_224\':\'ViT-S/16\',\'swin_tiny_patch4_window7_224\':\'Swin-Tiny\'}\nRESPONSES=[\'control_f1\',\'transfer_f1\',\'loss\']\n\n\ndef sha(p):\n    h=hashlib.sha256()\n    with open(p,\'rb\') as f:\n        for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\n\n\ndef helmert(n):\n    q=np.zeros((n,n));q[:,0]=1/np.sqrt(n)\n    for j in range(1,n):\n        q[:j,j]=1/np.sqrt(j*(j+1));q[j,j]=-j/np.sqrt(j*(j+1))\n    return q\n\n\ndef decompose(y,names):\n    """Exact orthogonal fixed-table decomposition on a complete balanced Cartesian grid."""\n    y=np.asarray(y,dtype=float)\n    if y.ndim!=len(names) or not np.isfinite(y).all():raise ValueError(\'Cuadrícula incompleta o no finita\')\n    axes=tuple(range(y.ndim));effects={():np.asarray(y.mean()).reshape((1,)*y.ndim)};rows=[]\n    centered=y-y.mean();total=float(np.sum(centered**2))\n    for order in range(1,y.ndim+1):\n        for subset in combinations(axes,order):\n            complement=tuple(a for a in axes if a not in subset)\n            effect=y.mean(axis=complement,keepdims=True) if complement else y.copy()\n            for size in range(order):\n                for proper in combinations(subset,size):effect=effect-effects[proper]\n            effects[subset]=effect\n            expanded=np.broadcast_to(effect,y.shape);ss=float(np.sum(expanded**2));df=math.prod(y.shape[a]-1 for a in subset)\n            rows.append(dict(term=\' × \'.join(names[a] for a in subset),order=order,df=df,ss=ss,\n                             mean_square=ss/df,percent_total_ss=100*ss/total if total>0 else np.nan,\n                             factors=\'|\'.join(names[a] for a in subset)))\n    reconstructed=sum(np.broadcast_to(e,y.shape) for e in effects.values())\n    error=float(np.max(np.abs(reconstructed-y)));ss_error=abs(sum(r[\'ss\'] for r in rows)-total)\n    # Independent orthonormal Helmert transform checks every component SS.\n    z=y.copy()\n    for axis,n in enumerate(y.shape):\n        z=np.moveaxis(np.tensordot(helmert(n).T,z,axes=(1,axis)),0,axis)\n    independent_errors=[]\n    for row in rows:\n        factors=row[\'factors\'].split(\'|\');slices=tuple(slice(1,None) if name in factors else slice(0,1) for name in names)\n        independent=float(np.sum(z[slices]**2));independent_errors.append(abs(independent-row[\'ss\']))\n    scale=max(1.,total)\n    if ss_error>1e-10*scale or error>1e-10 or max(independent_errors,default=0)>1e-10*scale:raise ValueError(\'Falla la descomposición independiente de SS\')\n    if sum(r[\'df\'] for r in rows)!=y.size-1:raise ValueError(\'Grados de libertad incorrectos\')\n    audit=dict(n_cells=y.size,total_df=y.size-1,total_ss=total,reconstruction_max_error=error,\n               ss_additivity_error=ss_error,helmert_max_ss_error=max(independent_errors,default=0),\n               residual_df=0,within_cell_replication=0)\n    return pd.DataFrame(rows),audit\n\n\ndef validate_zip(input_zip):\n    with ZipFile(input_zip) as z:\n        if z.testzip() is not None:raise ValueError(\'ZIP dañado\')\n        plan=json.loads(z.read(\'results/experiment_plan.json\'));status=json.loads(z.read(\'results/completion_summary.json\'))\n        if not status[\'complete\'] or status[\'completed_runs\']!=120 or status[\'evaluations\']!=240:raise ValueError(\'Experimento no está completo\')\n        signature=hashlib.sha256(json.dumps(plan,sort_keys=True).encode()).hexdigest();verified=0;markers=[]\n        for name in z.namelist():\n            if name.endswith(\'/COMPLETE.json\'):\n                markers.append(name);receipt=json.loads(z.read(name))\n                if receipt[\'signature\']!=signature:raise ValueError(\'Firma de plan diferente\')\n                parent=str(Path(name).parent)\n                for item in receipt[\'files\']:\n                    if item[\'area\']!=\'results\':continue\n                    if hashlib.sha256(z.read(parent+\'/\'+item[\'path\'])).hexdigest()!=item[\'sha256\']:raise ValueError(\'Resultados alterados\')\n                    verified+=1\n        if len(markers)!=120:raise ValueError(\'Faltan corridas completas\')\n        import io\n        d=pd.read_csv(io.BytesIO(z.read(\'results/fixed_target_losses_by_seed.csv\')))\n        # Verify all 120 pairs directly against their raw prediction tables, on identical image IDs.\n        from sklearn.metrics import f1_score\n        max_difference=0.\n        for row in d.itertuples():\n            prefix=f\'results/runs/match_{row.match_seed}/\'\n            other=\'OTHER\' if row.target==\'ADT45\' else \'ADT45\'\n            scores=[];frames=[]\n            for domain in [row.target,other]:\n                name=prefix+f\'train_{domain}__{row.model_key}__seed_{row.train_seed}/eval__{row.target}/predictions.csv\'\n                p=pd.read_csv(io.BytesIO(z.read(name))).sort_values(\'image_id\');frames.append(p)\n                scores.append(f1_score(p.label,p.y_pred_raw,labels=plan[\'class_names\'],average=\'macro\',zero_division=0))\n            if frames[0].image_id.tolist()!=frames[1].image_id.tolist():raise ValueError(\'Objetivos distintos\')\n            dif=max(abs(scores[0]-row.control_f1),abs(scores[1]-row.transfer_f1),abs(scores[0]-scores[1]-row.loss))\n            max_difference=max(max_difference,dif)\n        if max_difference>1e-12:raise ValueError(\'No se reproducen las pérdidas\')\n    return d,plan,dict(result_hashes_verified=verified,fixed_target_losses_reconstructed=120,max_score_difference=max_difference,\n                       input_zip_sha256=sha(input_zip),input_zip_size=Path(input_zip).stat().st_size,\n                       checkpoint_bytes_verified=False)\n\n\ndef analysis(input_zip,output):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    d,plan,verification=validate_zip(input_zip)\n    matches=plan[\'match_seeds\'];seeds=plan[\'train_seeds\'];frames=[];within=[];audits=[]\n    for target in [\'ADT45\',\'OTHER\']:\n        sub=d[d.target.eq(target)]\n        expected=pd.MultiIndex.from_product([MODELS,matches,seeds],names=[\'model_key\',\'match_seed\',\'train_seed\'])\n        indexed=sub.set_index([\'model_key\',\'match_seed\',\'train_seed\'])\n        if len(indexed)!=60 or indexed.index.duplicated().any() or set(indexed.index)!=set(expected):raise ValueError(\'Diseño no balanceado 4 × 5 × 3\')\n        indexed=indexed.loc[expected]\n        for response in RESPONSES:\n            values=indexed[response].to_numpy().reshape(4,5,3)\n            components,audit=decompose(values,[\'Configuration\',\'Matching\',\'Seed\'])\n            components.insert(0,\'response\',response);components.insert(0,\'target\',target);frames.append(components)\n            audits.append(dict(target=target,response=response,scope=\'four_configurations\',**audit))\n            for model in MODELS:\n                grid=sub[sub.model_key.eq(model)].set_index([\'match_seed\',\'train_seed\']).loc[pd.MultiIndex.from_product([matches,seeds])][response].to_numpy().reshape(5,3)\n                terms,audit=decompose(grid,[\'Matching\',\'Seed\']);terms.insert(0,\'model_key\',model);terms.insert(0,\'response\',response);terms.insert(0,\'target\',target);within.append(terms)\n                audits.append(dict(target=target,response=response,model_key=model,scope=\'within_configuration\',**audit))\n    global_terms=pd.concat(frames,ignore_index=True);local=pd.concat(within,ignore_index=True)\n    global_terms.to_csv(output/\'17_factorial_SS_by_target.csv\',index=False)\n    local.to_csv(output/\'17_within_configuration_SS.csv\',index=False)\n    pd.DataFrame(audits).to_csv(output/\'17_algebra_verification.csv\',index=False)\n    summaries=[]\n    for (target,response,model),g in local.groupby([\'target\',\'response\',\'model_key\']):\n        t=g.set_index(\'term\')\n        summaries.append(dict(target=target,response=response,model_key=model,\n            matching_percent=t.loc[\'Matching\',\'percent_total_ss\'],seed_main_percent=t.loc[\'Seed\',\'percent_total_ss\'],\n            matching_seed_interaction_percent=t.loc[\'Matching × Seed\',\'percent_total_ss\'],\n            within_matching_seed_spread_percent=t.loc[\'Seed\',\'percent_total_ss\']+t.loc[\'Matching × Seed\',\'percent_total_ss\']))\n    summary=pd.DataFrame(summaries);summary.to_csv(output/\'17_seed_and_matching_summary.csv\',index=False)\n    d.to_csv(output/\'17_verified_input_scores.csv\',index=False)\n    verification.update(dict(balanced_three_factor_tables=6,balanced_two_factor_tables=24,\n        max_reconstruction_error=max(a[\'reconstruction_max_error\'] for a in audits),\n        max_helmert_ss_error=max(a[\'helmert_max_ss_error\'] for a in audits),\n        no_F_statistics=True,no_p_values=True,no_random_effect_variance_estimates=True))\n    tests()\n    verification[\'synthetic_factor_tests_passed\']=True\n    (output/\'17_verification.json\').write_text(json.dumps(verification,indent=2)+\'\\n\')\n    config=dict(design=\'separate target domains; 4 model configurations × 5 matching constructions × 3 training seeds\',\n        responses=RESPONSES,definition=\'orthogonal sum-of-squares decomposition of a complete balanced table; percentage is SS_term / SS_total × 100\',\n        interactions=\'all main effects, all two-way interactions and the three-way interaction retained\',\n        degrees_of_freedom=\'59 total; 3,4,2,12,6,8,24 for configuration, matching, seed and their interactions\',\n        replication=\'one score per factor combination; saturated decomposition has zero residual df; highest interaction is not an independently estimated random error\',\n        scope=\'descriptive contributions to observed finite-table variation, not causal effects or population variance components\',\n        dependencies=\'overlapping matching image sets and common target images across configurations; no independence-based F inference\',\n        seed_scope=\'seed main effect can cancel across configurations or matchings; seed-related interactions are reported\',\n        pretraining=\'configurations have different pretrained lineages, so configuration contribution is not architecture-only\',\n        bootstrap_relation=\'bootstrap 16 estimates paired image-conditional uncertainty separately for each matching; ANOVA does not replace it\',\n        models=MODELS,match_seeds=matches,train_seeds=seeds,python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__)\n    (output/\'17_anova_config.json\').write_text(json.dumps(config,indent=2)+\'\\n\')\n    figures(output,global_terms,summary)\n    report(output,global_terms,summary,verification)\n    checksum(output)\n    print(json.dumps({\'output\':str(output),\'tables\':30,\'response\':\'macro-F1 control, transfer and loss\',\'F_tests\':False,\'p_values\':False}))\n    return global_terms,summary\n\n\ndef tests():\n    a=np.arange(4,dtype=float)[:,None,None];b=np.arange(5,dtype=float)[None,:,None];s=np.arange(3,dtype=float)[None,None,:]\n    cases=[(\'Configuration\',a+np.zeros((4,5,3))),(\'Matching\',b+np.zeros((4,5,3))),(\'Seed\',s+np.zeros((4,5,3))),\n           (\'Matching × Seed\',(b-b.mean())*(s-s.mean())+np.zeros((4,5,3))),\n           (\'Configuration × Matching × Seed\',(a-a.mean())*(b-b.mean())*(s-s.mean()))]\n    for term,y in cases:\n        table,_=decompose(y,[\'Configuration\',\'Matching\',\'Seed\']);row=table.set_index(\'term\').loc[term]\n        if not np.isclose(row.percent_total_ss,100,atol=1e-10):raise ValueError(\'Caso analítico no recuperado\')\n    table,audit=decompose(np.ones((4,5,3)),[\'Configuration\',\'Matching\',\'Seed\'])\n    if not table.percent_total_ss.isna().all() or audit[\'total_ss\']!=0:raise ValueError(\'Manejo incorrecto de respuesta constante\')\n\n\ndef figures(out,global_terms,summary):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    plt.rcParams.update({\'font.size\':11,\'axes.spines.top\':False,\'axes.spines.right\':False,\'pdf.fonttype\':42})\n    colors=[\'#2C6B92\',\'#CB7F30\',\'#3B8A76\',\'#9A5774\',\'#73819A\',\'#A6A43D\',\'#8C6852\']\n    terms=global_terms[(global_terms.response==\'loss\')&(global_terms.target==\'ADT45\')].term.tolist()\n    fig,ax=plt.subplots(figsize=(8.2,4.4));bottom=np.zeros(2)\n    short=[\'Configuration\',\'Matching\',\'Seed\',\'Configuration × Matching\',\'Configuration × Seed\',\'Matching × Seed\',\'Three-way interaction\']\n    for term,label,color in zip(terms,short,colors):\n        vals=[global_terms[(global_terms.response==\'loss\')&(global_terms.target==target)&(global_terms.term==term)].percent_total_ss.iloc[0] for target in [\'ADT45\',\'OTHER\']]\n        ax.bar([\'ADT45\',\'OTHER\'],vals,bottom=bottom,label=label,color=color,width=.48)\n        for x,v,b in zip(range(2),vals,bottom):\n            if v>=8:ax.text(x,b+v/2,f\'{v:.1f}%\',ha=\'center\',va=\'center\',color=\'white\',fontsize=10)\n        bottom+=vals\n    ax.set_ylim(0,100);ax.set_ylabel(\'Share of observed total SS (%)\');ax.set_xlabel(\'Target domain\')\n    ax.legend(loc=\'center left\',bbox_to_anchor=(1.02,.5),frameon=False,fontsize=9);fig.tight_layout()\n    for ext in [\'png\',\'pdf\']:fig.savefig(out/f\'17_factorial_loss_decomposition.{ext}\',dpi=300,bbox_inches=\'tight\')\n    plt.close(fig)\n    fig,axes=plt.subplots(1,2,figsize=(9.6,4.4),sharex=True)\n    cols=[\'matching_percent\',\'seed_main_percent\',\'matching_seed_interaction_percent\'];labels=[\'Matching\',\'Seed main effect\',\'Matching × seed\']\n    for ax,target in zip(axes,[\'ADT45\',\'OTHER\']):\n        d=summary[(summary.response==\'loss\')&(summary.target==target)].set_index(\'model_key\').loc[MODELS];left=np.zeros(4)\n        for col,label,color in zip(cols,labels,[\'#CB7F30\',\'#3B8A76\',\'#73819A\']):\n            ax.barh([LABELS[m] for m in MODELS],d[col],left=left,label=label,color=color)\n            for y,v,l in zip(range(4),d[col],left):\n                if v>=12:ax.text(l+v/2,y,f\'{v:.0f}%\',ha=\'center\',va=\'center\',color=\'white\',fontsize=9)\n            left+=d[col].to_numpy()\n        ax.set_title(\'Target \'+target);ax.set_xlim(0,100);ax.set_xlabel(\'Share of within-configuration SS (%)\');ax.invert_yaxis()\n    handles,labels=axes[0].get_legend_handles_labels();fig.legend(handles,labels,loc=\'lower center\',bbox_to_anchor=(.5,-.02),ncol=3,frameon=False,fontsize=9)\n    fig.tight_layout(rect=(0,.07,1,1))\n    for ext in [\'png\',\'pdf\']:fig.savefig(out/f\'17_matching_seed_decomposition.{ext}\',dpi=300,bbox_inches=\'tight\')\n    plt.close(fig)\n\n\ndef report(out,global_terms,summary,v):\n    lines=[\'# Descomposición ANOVA descriptiva de P5\',\'\',\n        \'Se añadió un diagnóstico de variabilidad a las 120 corridas nuevas. Complementa los intervalos bootstrap; no utiliza pruebas F ni valores p que supongan independencia de los emparejamientos.\',\n        \'\', \'## Qué se descompone\',\'\',\n        \'Por cada dominio objetivo se organiza una tabla balanceada de 60 puntuaciones: cuatro configuraciones, cinco construcciones emparejadas y tres semillas. Se analizan por separado el macro-F1 de control, el macro-F1 transferido y su pérdida sobre las mismas imágenes objetivo.\',\n        \'\', \'La descomposición ortogonal conserva los tres efectos principales, las tres interacciones de dos factores y la interacción triple. Sus sumas de cuadrados suman la variación total observada. Los porcentajes no son efectos causales ni estimaciones de componentes de varianza poblacional.\',\n        \'\', \'## Pérdida de macro-F1\',\'\', \'| Componente | gl | Objetivo ADT45 % SS | Objetivo OTHER % SS |\',\'| --- | ---: | ---: | ---: |\']\n    g=global_terms[global_terms.response.eq(\'loss\')]\n    for term in g.term.drop_duplicates():\n        d=g[g.term.eq(term)].set_index(\'target\');lines.append(f\'| {term} | {int(d.df.iloc[0])} | {d.loc["ADT45","percent_total_ss"]:.2f} | {d.loc["OTHER","percent_total_ss"]:.2f} |\')\n    lines+=[\'\', \'## Semillas dentro de cada configuración\',\'\',\n        \'El efecto principal de semilla mide un patrón promedio compartido entre construcciones. Puede ser pequeño aunque algunas semillas difieran mucho dentro de cada construcción. Por eso se informa también la interacción emparejamiento × semilla. La suma de ambos términos equivale a la variación entre las tres semillas dentro de los cinco emparejamientos.\',\n        \'\', \'| Configuración | Objetivo | Emparejamiento % | Semilla principal % | Interacción % | Semilla e interacción % |\',\'| --- | --- | ---: | ---: | ---: | ---: |\']\n    for r in summary[summary.response.eq(\'loss\')].itertuples():\n        lines.append(f\'| {LABELS[r.model_key]} | {r.target} | {r.matching_percent:.2f} | {r.seed_main_percent:.2f} | {r.matching_seed_interaction_percent:.2f} | {r.within_matching_seed_spread_percent:.2f} |\')\n    lines+=[\'\',\'## Límites de interpretación\',\'\',\n        \'Hay una puntuación por combinación de factores. Al conservar todas las interacciones, el modelo es saturado: 59 grados de libertad de variación y cero grados de libertad residuales. La interacción triple no es una estimación independiente del error aleatorio.\',\n        \'\', \'Las construcciones comparten imágenes y las configuraciones se evalúan sobre los mismos objetivos. No se calcula un F convencional, ni valores p, ni se declara que el porcentaje de semilla estime la varianza de cualquier futuro entrenamiento. Las configuraciones tienen procedencias de preentrenamiento diferentes; el efecto de configuración no aísla solamente arquitectura.\',\n        \'\', \'Las tres semillas no son muestras de campo nuevas. La variación entre cinco construcciones se conserva como sensibilidad interna; no representa cinco poblaciones independientes. El resultado principal de transferencia sigue sustentándose en las diferencias pareadas y el bootstrap 16.\',\n        \'\', \'## Verificación\',\'\',\n        f\'Se comprobaron las huellas de {v["result_hashes_verified"]} archivos de resultados y se reconstruyeron las 120 pérdidas desde las predicciones, con discrepancia máxima {v["max_score_difference"]:.3g}. Las 30 descomposiciones se verificaron con una transformación ortonormal de Helmert independiente, pruebas sintéticas de efectos puros e interacciones y sumas de grados de libertad. Discrepancia máxima en SS: {v["max_helmert_ss_error"]:.3g}.\',\n        \'\', \'## Texto para el manuscrito\',\'\',\n        \'**Methods.** We added a descriptive factorial ANOVA decomposition of the observed macro-F1 scores and fixed-target losses, separately for each target domain. The balanced 4-configuration × 5-matching × 3-seed table was decomposed orthogonally into all main effects, all two-way interactions and the three-way interaction. Component contributions were reported as percentages of total corrected sum of squares. A complementary 5-matching × 3-seed decomposition within each configuration separated matching, seed and matching-by-seed terms. With one score per factor combination, the full decomposition had zero residual degrees of freedom. We therefore reported no F statistics, p-values or population variance components. This diagnostic did not replace the paired bootstrap and did not treat overlapping matching constructions as independent cohorts.\',\n        \'\', \'El paquete contiene los CSV de todos los términos para control, transferencia y pérdida, las figuras PNG/PDF, las verificaciones y el código reproducible.\']\n    (out/\'17_Informe_ANOVA_Descriptiva_P5.md\').write_text(\'\\n\'.join(lines)+\'\\n\')\n\n\ndef checksum(out):\n    with (Path(out)/\'SHA256.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.writer(f);w.writerow([\'relative_path\',\'size_bytes\',\'sha256\'])\n        for p in sorted(Path(out).rglob(\'*\')):\n            if p.is_file() and p.name!=\'SHA256.csv\':w.writerow([p.relative_to(out).as_posix(),p.stat().st_size,sha(p)])\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'--input-zip\',required=True);parser.add_argument(\'--output\',required=True)\n    args=parser.parse_args();analysis(args.input_zip,args.output)\n'
MODULE_PATH = Path('/content/paddy17_anova_descriptiva.py')
MODULE_PATH.write_text(MODULE_TEXT, encoding='utf-8')
spec = importlib.util.spec_from_file_location('paddy17_anova_descriptiva', MODULE_PATH)
p17 = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = p17
spec.loader.exec_module(p17)


## 3. Calcular la descomposición
El porcentaje es **SS del componente / SS total × 100**. Describe variación de estas puntuaciones, no el porcentaje de la pérdida causado por un factor.

Un efecto principal de semilla pequeño puede coexistir con interacciones grandes. Por eso se informa una segunda descomposición dentro de cada configuración: emparejamiento, semilla e interacción emparejamiento × semilla. La suma de los dos últimos términos expresa la variación entre semillas dentro de los emparejamientos observados.


In [ ]:
factorial, seed_matching = p17.analysis(INPUT_ZIP, OUTPUT)
shutil.copyfile(MODULE_PATH, OUTPUT / MODULE_PATH.name)
p17.checksum(OUTPUT)
print('ANOVA descriptiva terminada; sin entrenamiento adicional.')


## 4. Revisar resultados y figuras
La pérdida sigue siendo control menos transferencia en las mismas imágenes objetivo. El efecto de configuración incluye sus diferencias de preentrenamiento; no aísla solamente la arquitectura.


In [ ]:
from IPython.display import display, Image, Markdown
display(factorial.query("response == 'loss'")[['target','term','df','ss','percent_total_ss']])
display(seed_matching.query("response == 'loss'"))
display(Image(filename=str(OUTPUT / '17_factorial_loss_decomposition.png')))
display(Image(filename=str(OUTPUT / '17_matching_seed_decomposition.png')))
display(Markdown((OUTPUT / '17_Informe_ANOVA_Descriptiva_P5.md').read_text()))


## 5. Exportar el diagnóstico
El paquete de salida incluye las tablas de los tres desenlaces, las figuras, configuración, verificaciones y código. No contiene imágenes ni pesos.


In [ ]:
EXPORT = PROJECT / 'exports/17_P5_ANOVA_Descriptiva_RESULTADOS'
zip_path = shutil.make_archive(str(EXPORT), 'zip', root_dir=OUTPUT)
print('Archivo creado:', zip_path)
# Descarga opcional:
# from google.colab import files
# files.download(zip_path)
